# Notebook 03 — Anomaly Detection

Train on **non-toxic (class 0)** data only; score all validation messages as anomalies. Metric: **AUROC** (threshold-free discrimination).

In [ ]:
# Anomaly detection: train on non-toxic class 0 data only, score toxic as anomalies.
# TruncatedSVD required — IsolationForest/OneClassSVM degrade in high-dim sparse TF-IDF space.
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import normalize
import sys
sys.path.insert(0, str(Path('..').resolve()))
from src.loaders import load_wot, load_dota, load_combined
from src.scoring import append_registry
from src.label_schemes import WOT_SCHEMES, DOTA_SCHEMES, WOT_CLASS_NAMES, DOTA_CLASS_NAMES

CONFIG = {
    'seed': 7524,
    'text_col': 'clean_message',
    'label_col': 'label',
    'svd_components': 100,
    'registry_path': Path('../data/results/results_registry.csv'),
}
seed = CONFIG['seed']
print('CONFIG loaded.')

**Pipeline:** TF-IDF → TruncatedSVD (LSA, 100 components) → anomaly model (IsolationForest or OneClassSVM).

Training uses **only class 0** (non-toxic) messages — the model learns the "normal" distribution. At inference, toxic messages should appear as anomalies (high anomaly score). AUROC measures how well the model separates class 0 from all toxic classes without requiring any label-based threshold tuning.

## Section 1 — Dimensionality Reduction Check

In [ ]:
# Verify TruncatedSVD(100) retains >= 70% variance before running anomaly models.
# If < 70%, svd_components should be increased to 200 in CONFIG.
combined_train = load_combined('train', wot_scheme=WOT_SCHEMES[2], dota_scheme=DOTA_SCHEMES[2])
tfidf_all = TfidfVectorizer(ngram_range=(1,2), min_df=3, max_df=0.95,
                              sublinear_tf=True, norm='l2')
X_combined_tfidf = tfidf_all.fit_transform(combined_train[CONFIG['text_col']])
print(f'TF-IDF vocab size: {X_combined_tfidf.shape[1]:,}')

svd = TruncatedSVD(n_components=CONFIG['svd_components'], random_state=seed)
svd.fit(X_combined_tfidf)
explained = svd.explained_variance_ratio_.sum()
print(f'Explained variance at {CONFIG["svd_components"]} components: {explained:.3f}')
if explained < 0.70:
    print('⚠ < 70% — consider increasing svd_components to 200 in CONFIG')
else:
    print('✓ Sufficient variance retained')

**Why dimensionality reduction is required:** TF-IDF on bigrams produces a sparse matrix with tens of thousands of dimensions. IsolationForest and OneClassSVM both degrade severely in high-dimensional spaces — the "curse of dimensionality" causes all pairwise distances to converge, making anomaly scores uninformative. TruncatedSVD (LSA) projects the sparse matrix into a dense 100-dimensional semantic space, where proximity is meaningful and anomaly models can find meaningful boundaries around the "normal" (non-toxic) cluster.

## Section 2 — Anomaly Detection Setups

In [ ]:
# Three setups x two models = 6 experiments.
# Training data: class 0 only (non-toxic, already cleaned - no conflicting labels).
# Metric: AUROC - no threshold needed, measures discrimination directly.

def build_anomaly_pipe(n_components: int, model_name: str, seed: int):
    """TF-IDF -> TruncatedSVD -> anomaly model pipeline."""
    tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=3, max_df=0.95,
                             sublinear_tf=True, norm='l2')
    svd   = TruncatedSVD(n_components=n_components, random_state=seed)
    if model_name == 'IsolationForest':
        detector = IsolationForest(contamination=0.05, random_state=seed, n_jobs=-1)
    else:
        detector = OneClassSVM(kernel='rbf', nu=0.05)
    return Pipeline([('tfidf', tfidf), ('svd', svd), ('detector', detector)])


def run_anomaly_setup(train_texts: pd.Series, val_texts: pd.Series,
                       val_labels_binary: pd.Series, setup_name: str,
                       train_game: str, test_game: str, n_components: int,
                       registry_path: Path):
    results = {}
    for model_name in ['IsolationForest', 'OneClassSVM']:
        pipe = build_anomaly_pipe(n_components, model_name, seed)
        pipe.fit(train_texts)
        # Negate decision_function: higher = more anomalous (toxic)
        scores = -pipe.decision_function(val_texts)
        auroc  = roc_auc_score(val_labels_binary, scores)
        print(f'  {setup_name} | {model_name:<18} AUROC={auroc:.4f}')
        results[model_name] = {'scores': scores, 'auroc': auroc}

        append_registry({
            'experiment':       'anomaly_detection',
            'train_game':       train_game,
            'test_game':        test_game,
            'n_classes':        2,
            'label_scheme':     'binary',
            'model':            model_name,
            'cv_macro_f1':      None, 'cv_std': None, 'cv_weighted_f1': None,
            'test_macro_f1':    None, 'test_weighted_f1': None,
            'per_class_recall': None,
            'ood_macro_f1':     None, 'ood_weighted_f1': None,
            'anomaly_auroc':    round(auroc, 4),
            'notes':            setup_name,
        }, path=registry_path)
    return results

n_comp = CONFIG['svd_components']
reg_path = CONFIG['registry_path']

# Setup 1: WoT class 0 -> WoT val
wot_train  = load_wot('train')
wot_val    = load_wot('val')
wot_train0 = wot_train[wot_train[CONFIG['label_col']]==0][CONFIG['text_col']]
wot_val_bin = (wot_val[CONFIG['label_col']].astype(int) > 0).astype(int)
print('=== Setup 1: Train WoT class 0 -> Test WoT val ===')
setup1 = run_anomaly_setup(wot_train0, wot_val[CONFIG['text_col']],
                            wot_val_bin, 'wot_on_wot', 'WoT', 'WoT', n_comp, reg_path)

# Setup 2: Dota class 0 -> Dota val
dota_train  = load_dota('train')
dota_val    = load_dota('val')
dota_train0 = dota_train[dota_train[CONFIG['label_col']]==0][CONFIG['text_col']]
dota_val_bin = (dota_val[CONFIG['label_col']].astype(int) > 0).astype(int)
print('\n=== Setup 2: Train Dota class 0 -> Test Dota val ===')
setup2 = run_anomaly_setup(dota_train0, dota_val[CONFIG['text_col']],
                            dota_val_bin, 'dota_on_dota', 'Dota', 'Dota', n_comp, reg_path)

# Setup 3: Combined class 0 -> each game
combined0 = pd.concat([wot_train0, dota_train0], ignore_index=True)
print('\n=== Setup 3a: Combined class 0 -> WoT val ===')
setup3a = run_anomaly_setup(combined0, wot_val[CONFIG['text_col']],
                             wot_val_bin, 'combined_on_wot', 'WoT+Dota', 'WoT', n_comp, reg_path)
print('\n=== Setup 3b: Combined class 0 -> Dota val ===')
setup3b = run_anomaly_setup(combined0, dota_val[CONFIG['text_col']],
                             dota_val_bin, 'combined_on_dota', 'WoT+Dota', 'Dota', n_comp, reg_path)

**Interpreting AUROC results:**

| AUROC | Meaning |
|-------|--------|
| > 0.80 | Strong signal — anomaly model reliably separates toxic from non-toxic |
| 0.65–0.80 | Useful signal — worth including in ensemble |
| ~0.50 | No signal — model cannot distinguish toxicity without labels |

**Combined training hypothesis:** Pooling WoT + Dota class 0 messages exposes the model to broader "normal" vocabulary. If toxic language is game-specific (e.g., Dota hero names used as insults), a combined model may generalise better than a game-specific one — or it may dilute the signal if non-toxic language varies too much between games.

## Section 3 — Anomaly Score Distribution per Class

In [ ]:
# Box plot of anomaly scores per original class reveals which toxic categories
# are hardest to detect without labels. Expected: Non-Toxic low, Extremism high.
pipe_wot = build_anomaly_pipe(n_comp, 'IsolationForest', seed)
pipe_wot.fit(wot_train0)
wot_val_scores = -pipe_wot.decision_function(wot_val[CONFIG['text_col']])

pipe_dota = build_anomaly_pipe(n_comp, 'IsolationForest', seed)
pipe_dota.fit(dota_train0)
dota_val_scores = -pipe_dota.decision_function(dota_val[CONFIG['text_col']])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (val_df, scores, game, cn) in zip(axes, [
    (wot_val,  wot_val_scores,  'WoT',  WOT_CLASS_NAMES[6]),
    (dota_val, dota_val_scores, 'Dota', DOTA_CLASS_NAMES[4]),
]):
    plot_df = pd.DataFrame({'score': scores, 'class': val_df[CONFIG['label_col']].values})
    plot_df['class_name'] = plot_df['class'].map({i: n for i, n in enumerate(cn)})
    sns.boxplot(data=plot_df, x='class_name', y='score', ax=ax,
                order=cn, palette='YlOrRd')
    ax.set_title(f'{game} — Anomaly Score per Class (IsolationForest)', fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Anomaly Score (higher = more anomalous)')
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
Path('../data/results').mkdir(parents=True, exist_ok=True)
plt.savefig('../data/results/anomaly_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:**

- **Ideal pattern:** Scores increase monotonically from Non-Toxic (lowest) through mild toxicity up to Extremism/Severe (highest). This confirms LSA-compressed features capture toxicity intensity without labels.
- **Flat pattern:** All classes show similar score distributions — LSA collapsed game-specific toxic vocabulary into shared semantic dimensions, making categories indistinguishable. This motivates using supervised classifiers rather than anomaly detection for fine-grained classification.
- **Partial signal:** Non-Toxic clearly separates from the most extreme category, but middle classes overlap — useful for binary toxic/non-toxic detection but insufficient for multi-class.